In [1]:
import model
import data

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader

from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
# device="mps"
## Load Dataset
train_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_train.npy", ks_npy="../Data/ks_train.npy")
val_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_val.npy", ks_npy="../Data/ks_val.npy")
test_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_test.npy", ks_npy="../Data/ks_test.npy")

# train_dataset = train_dataset.to(device)
# val_dataset = val_dataset.to(device)
# test_dataset = test_dataset.to(device)

train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=100, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False)

In [ ]:
import os
import time
import json
import csv
import itertools
import torch
import torch.nn as nn
from torch.optim import Adam
from tqdm import tqdm
import model 

# ==========================================
#              CONFIG & SETUP
# ==========================================
LRS = [1e-3]
NUM_LAYERS = [2, 3, 4, 5]
P_VALUES = [8, 16, 32, 64]
WEIGHT_DECAYS = [0.0] 

MAX_EPOCHS = 1000
PATIENCE = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("histories", exist_ok=True)
catalog_file = "master_catalog.csv"

if not os.path.exists(catalog_file):
    with open(catalog_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Run_Name", "LR", "Num_Layers", "p", "Weight_Decay", "Best_Val_Loss", "Epochs_Trained"])

# ==========================================
#           TRAINING FUNCTION
# ==========================================
def train_config(lr, nl, p, wd, train_loader, val_loader):
    run_name = f"model_lr{lr}_nl{nl}_p{p}_wd{wd}"
    print(f"\n{'-'*50}\nSTARTING RUN: {run_name}\n{'-'*50}")
    
    m = model.OperatorNetwork(trunk_input_size=7, branch_input_size=6, num_layers=nl, p=p).to(device)
    criterion = nn.MSELoss()
    optimizer = Adam(m.parameters(), lr=lr, weight_decay=wd)
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_epoch = 0
    
    history = {'train_loss': [], 'val_loss': []}
    checkpoint_path = f"checkpoints/{run_name}_best.pth"
    
    # Single progress bar for the entire run
    epoch_bar = tqdm(range(MAX_EPOCHS), desc=run_name, leave=True)
    
    for e in epoch_bar:
        
        # --- TRAIN ---
        m.train()
        running_train_loss = 0.0
        
        for batch in train_loader:
            branch_input, trunk_input, target = [b.to(device) for b in batch]

            pred = m(seq=trunk_input, A=branch_input)
            loss = criterion(pred, target)

            optimizer.zero_grad() 
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
            
            optimizer.step()

            running_train_loss += loss.item() * target.size(0)
            
        avg_train_loss = running_train_loss / len(train_loader.dataset)
        history['train_loss'].append(avg_train_loss)

        # --- VALIDATION ---
        m.eval()
        running_val_loss = 0.0
        
        with torch.no_grad():
            for batch in val_loader:
                branch_input, trunk_input, target = [b.to(device) for b in batch]
                
                pred = m(seq=trunk_input, A=branch_input)
                val_loss = criterion(pred, target)
                
                running_val_loss += val_loss.item() * target.size(0)
                
        avg_val_loss = running_val_loss / len(val_loader.dataset)
        history['val_loss'].append(avg_val_loss)
        
        # --- EARLY STOPPING & CHECKPOINTING ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = e
            epochs_no_improve = 0
            
            torch.save({
                'epoch': e,
                'model_state_dict': m.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
            }, checkpoint_path)
        else:
            epochs_no_improve += 1

        # Update the progress bar text dynamically
        epoch_bar.set_postfix({
            'Train Loss': f"{avg_train_loss:.6f}",
            'Val Loss': f"{avg_val_loss:.6f}",
            'Best Val': f"{best_val_loss:.6f}",
            'Patience': f"{epochs_no_improve}/{PATIENCE}"
        })
            
        if epochs_no_improve >= PATIENCE:
            # Use epoch_bar.write to print without breaking the progress bar visualization
            epoch_bar.write(f"Early stopping triggered! No improvement for {PATIENCE} epochs.")
            break

    # Close the progress bar cleanly when the run finishes
    epoch_bar.close()

    # --- POST-RUN LOGGING ---
    with open(f"histories/{run_name}_history.json", "w") as f:
        json.dump(history, f)
        
    with open(catalog_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([run_name, lr, nl, p, wd, best_val_loss, best_epoch + 1])
        
    print(f"Finished {run_name}. Best Val Loss: {best_val_loss:.6f} at Epoch {best_epoch+1}")
    
    del m, optimizer, criterion
    torch.cuda.empty_cache()

# ==========================================
#              GRID SEARCH LOOP
# ==========================================
hyperparameter_combinations = list(itertools.product(LRS, NUM_LAYERS, P_VALUES, WEIGHT_DECAYS))

print(f"Total configurations to test: {len(hyperparameter_combinations)}")

for lr, nl, p, wd in hyperparameter_combinations:
    
    run_exists = False
    if os.path.exists(catalog_file):
        with open(catalog_file, mode='r') as f:
            reader = csv.reader(f)
            next(reader) 
            for row in reader:
                if row[0] == f"model_lr{lr}_nl{nl}_p{p}_wd{wd}":
                    run_exists = True
                    break
                    
    if run_exists:
        print(f"Skipping model_lr{lr}_nl{nl}_p{p}_wd{wd} (already in catalog)")
        continue
        
    train_config(lr, nl, p, wd, train_loader, val_loader)

print("ALL HYPERPARAMETER SWEEPS COMPLETE!")

Total configurations to test: 16
Skipping model_lr0.001_nl2_p8_wd0.0 (already in catalog)
Skipping model_lr0.001_nl2_p16_wd0.0 (already in catalog)
Skipping model_lr0.001_nl2_p32_wd0.0 (already in catalog)
Skipping model_lr0.001_nl2_p64_wd0.0 (already in catalog)

--------------------------------------------------
STARTING RUN: model_lr0.001_nl3_p8_wd0.0
--------------------------------------------------


model_lr0.001_nl3_p8_wd0.0:  31%|███       | 310/1000 [08:08<18:07,  1.58s/it, Train Loss=0.000276, Val Loss=0.000368, Best Val=0.000312, Patience=50/50]


Early stopping triggered! No improvement for 50 epochs.
Finished model_lr0.001_nl3_p8_wd0.0. Best Val Loss: 0.000312 at Epoch 261

--------------------------------------------------
STARTING RUN: model_lr0.001_nl3_p16_wd0.0
--------------------------------------------------


model_lr0.001_nl3_p16_wd0.0:  26%|██▋       | 264/1000 [07:27<20:39,  1.68s/it, Train Loss=0.000144, Val Loss=0.000214, Best Val=0.000154, Patience=18/50]